In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    concat_ws,
    current_timestamp,
    date_format,
    from_utc_timestamp,
    length,
    lit,
    sha2,
    to_date,
    when
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
dbutils.widgets.text('externalfiles_zone', 'None', '07-Externalfiles Zone')
externalfiles_zone = dbutils.widgets.get('externalfiles_zone')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_ext_union_file_extfile(
    spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        # -------DATE_FILTER
        date_filter = f"businessdate = {run_date!r}"

        # -------SCHEMA
        # schema = (
        #       "ExtUnionFileId string,"
        #       "SourceSystem string,"
        #       "PortfolioKey string,"
        #       "PortfolioName string,"
        #       "IngestionTimestamp timestamp,"
        #       "BatchId string,"
        #       "ExtUnionFileKey string,"
        #       "Groupkey string,"
        #       "GroupName string,"
        #       "BorrowerKey string,"
        #       "BorrowerName string,"
        #       "BorrowerTaxIdentificationNumber string,"
        #       "MonitoringUnitCode string,"
        #       "ObligorRatingKey string,"
        #       "ObligorRatingDate string,"
        #       "ObligorCountry string,"
        #       "UniqueExposureKey string,"
        #       "ExposureKey string,"
        #       "ExposureNumber string,"
        #       "ExposureSourceSystemKey string,"
        #       "ExposureCurrency string,"
        #       "ExposureSegmentation string,"
        #       "ProductCode string,"
        #       "ProductType string,"
        #       "IFRSGrossCarryingAmount string,"
        #       "SuspendedInterest string,"
        #       "AccruedSuspendedInterest string,"
        #       "AccruedInterest string,"
        #       "OnbalanceAmount string,"
        #       "TotalRemainingForAmortization string,"
        #       "BalanceOfPendingTransactions string,"
        #       "OffBalanceAmount string,"
        #       "UndrawnAmount string,"
        #       "ImpairedFlag string,"
        #       "ImpairmentType string,"
        #       "IFRS9Stage string,"
        #       "ImpairmentAllowance string,"
        #       "FairValueAdjustmentsCR string,"
        #       "ProvisionsL3869 string,"
        #       "TotalProvisionsOnBalance string,"
        #       "ProvisionsOffBalance string,"
        #       "ProvisionsUndrawnLimits string,"
        #       "FairValueAdjustmentsFVPL string,"
        #       "IFRS9ClassificationFlag string,"
        #       "IFRS9POCIFlag string,"
        #       "EliminationFlag string,"
        #       "ServiceTypeCode string,"
        #       "ServiceType string,"
        #       "BookingEntity string,"
        #       "OriginatingEntity string,"
        #       "DaysPastDue string,"
        #       "CurrentAmount string,"
        #       "DelinquentAmount string,"
        #       "DelinquentAmountBucket1 string,"
        #       "DelinquentAmountBucket2 string,"
        #       "DelinquentAmountBucket3 string,"
        #       "DelinquentAmountBucket4 string,"
        #       "DelinquentAmountBucket5 string,"
        #       "DelinquentAmountBucket6 string,"
        #       "DelinquentAmountBucket7 string,"
        #       "DelinquentAmountBucket8 string,"
        #       "DelinquentAmountBucket9 string,"
        #       "DelinquentAmountBucket10 string,"
        #       "DelinquentAmountBucket11 string,"
        #       "DelinquentAmountBucket12 string,"
        #       "DelinquentAmountBucket13 string,"
        #       "DelinquentAmountBucket14 string,"
        #       "BucketCategoryNMateriality string,"
        #       "BucketCategoryWithMateriality string,"
        #       "NPLNoMaterialityFlag string,"
        #       "NPLWithMaterialityFlag string,"
        #       "EBAStatus string,"
        #       "UTPTypeCode string,"
        #       "UTPType string,"
        #       "ProbationStartDate string,"
        #       "ECAAssetClass string,"
        #       "FINREPAssetClassCode string,"
        #       "FINREPAssetClass string,"
        #       "IFRSAssetClass string,"
        #       "ForbearanceTypeCode string,"
        #       "ForbearanceType string,"
        #       "ForbearanceDate string,"
        #       "NaceCode string,"
        #       "ECAIndustry string,"
        #       "FINREPIndustry string,"
        #       "IFRSIndustry string,"
        #       "ECAPublicFlag string,"
        #       "StateGuaranteeFlag string,"
        #       "CollateralCapCash string,"
        #       "CollateralCapRRE string,"
        #       "CollateralCapCRE string,"
        #       "CollateralCapORE string,"
        #       "CollateralCapVessels string,"
        #       "CollateralCapTbills string,"
        #       "CollateralCapGGG string,"
        #       "CollateralCapStateRec string,"
        #       "CollateralCapInventory string,"
        #       "CollateralCapAssRec string,"
        #       "CollateralCapIndustrial string,"
        #       "CollateralCapOtherCollateral string,"
        #       "CollateralCapOtherFinGuarantee string,"
        #       "CollateralCapTotalTangible string,"
        #       "CollateralCapCorporateGuarantee string,"
        #       "CollateralCapPersonalGuarantee string,"
        #       "HFSFlag string,"
        #       "SPVsCapitalization string,"
        #       "DeNuncementFlag string,"
        #       "LTV string,"
        #       "LTVRE string,"
        #       "ForbearanceMeasuresCount string,"
        #       "ForbearanceOfNPEFlag string,"
        #       "FPLUnderProbationFlag string,"
        #       "AgreementKey string,"
        #       "CREStocktakeFlag string,"
        #       "FINREPAssetClassSubClassCode string,"
        #       "FinancialDifficultyFlag string,"
        #       "InterestRate string,"
        #       "EffectiveInterestRate string,"
        #       "AcMaintenServiceBranchCode string,"
        #       "AcMaintenServiceBranch string,"
        #       "RelationshipUnitCode string,"
        #       "RelationshipUnit string,"
        #       "ConnectedBorrowerCategoryCode string,"
        #       "ConnectedBorrowerCategory string,"
        #       "ExposureMaturityDate string,"
        #       "ObligorLimitEndDate string,"
        #       "LitigationFlag string,"
        #       "FinRepCre string,"
        #       "BucketGt1Last12MFlag string,"
        #       "BucketGt1Last12MDate string,"
        #       "ObligorLimit string,"
        #       "SecuritizationCode string,"
        #       "MainBorrowerKey string,"
        #       "MainBorrowerTaxIdentificationNumber string,"
        #       "TepixCovidFlag string,"
        #       "NonCoreFlag string,"
        #       "PastDueDaysObligor string,"
        #       "PastDueDaysFacility string,"
        #       "DefaultStatus string,"
        #       "DefaultStatusReasonCode string,"
        #       "DefaultStatusReason string,"
        #       "DefaultDate string,"
        #       "NPVLossOver1PercentFlag string,"
        #       "EliminatedEntityCode string,"
        #       "PBOriginationDate string,"
        #       "CPRSLevel1 string,"
        #       "CPRSLevel2 string,"
        #       "SnapshotDate date,"
        #       "BusinessDate date"
        #   )

        cluster_columns = ["BusinessDate"]

        # -------CHOOSE PATH

        curated_table_path = f"{curated_zone}.ExtUnionFile"

        # -------CREATION OF DELTA TABLE
        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "extfile",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # --------APPLY FILTERS AND SELECT COLUMNS FROM THE TABLES WE NEED

        # ================== UNION_FILE ==================

        union_file_extra_filters = None

        if union_file_extra_filters is not None:
            union_file_extra_filters.append(date_filter)
            union_file_final_filters = " AND ".join(union_file_extra_filters)
        else:
            union_file_final_filters = date_filter

        union_file_selected_fields = [
            "SnapshotDate",
            "Groupkey",
            "GroupName",
            "BorrowerKey",
            "BorrowerName",
            "BorrowerTaxIdentificationNumber",
            "MonitoringUnitCode",
            "ObligorRatingKey",
            "ObligorRatingDate",
            "ObligorCountry",
            "UniqueExposureKey",
            "ExposureKey",
            "ExposureNumber",
            "ExposureSourceSystemKey",
            "ExposureCurrency",
            "ExposureSegmentation",
            "ProductCode",
            "ProductType",
            "IFRSGrossCarryingAmount",
            "SuspendedInterest",
            "AccruedSuspendedInterest",
            "AccruedInterest",
            "OnbalanceAmount",
            "TotalRemainingForAmortization",
            "BalanceOfPendingTransactions",
            "OffBalanceAmount",
            "UndrawnAmount",
            "ImpairedFlag",
            "ImpairmentType",
            "IFRS9Stage",
            "ImpairmentAllowance",
            "FairValueAdjustmentsCR",
            "ProvisionsL3869",
            "TotalProvisionsOnBalance",
            "ProvisionsOffBalance",
            "ProvisionsUndrawnLimits",
            "FairValueAdjustmentsFVPL",
            "IFRS9ClassificationFlag",
            "IFRS9POCIFlag",
            "EliminationFlag",
            "ServiceType",
            "BookingEntity",
            "OriginatingEntity",
            "DaysPastDue",
            "CurrentAmount",
            "DelinquentAmount",
            "DelinquentAmountBucket1",
            "DelinquentAmountBucket2",
            "DelinquentAmountBucket3",
            "DelinquentAmountBucket4",
            "DelinquentAmountBucket5",
            "DelinquentAmountBucket6",
            "DelinquentAmountBucket7",
            "DelinquentAmountBucket8",
            "DelinquentAmountBucket9",
            "DelinquentAmountBucket10",
            "DelinquentAmountBucket11",
            "DelinquentAmountBucket12",
            "DelinquentAmountBucket13",
            "DelinquentAmountBucket14",
            "BucketCategoryNMateriality",
            "BucketCategoryWithMateriality",
            "NPLNoMaterialityFlag",
            "NPLWithMaterialityFlag",
            "EBAStatus",
            "UTPIndicator",
            "ProbationStartDate",
            "ECAAssetClass",
            "FINREPAssetClass",
            "IFRSAssetClass",
            "ForbearanceType",
            "ForbearanceDate",
            "NaceCode",
            "ECAIndustry",
            "FINREPIndustry",
            "IFRSIndustry",
            "ECAPublicFlag",
            "StateGuaranteeFlag",
            "CollateralCapCash",
            "CollateralCapRRE",
            "CollateralCapCRE",
            "CollateralCapORE",
            "CollateralCapVessels",
            "CollateralCapTbills",
            "CollateralCapGGG",
            "CollateralCapStateRec",
            "CollateralCapInventory",
            "CollateralCapAssRec",
            "CollateralCapIndustrial",
            "CollateralCapOtherCollateral",
            "CollateralCapOtherFinGuarantee",
            "CollateralCapTotalTangible",
            "CollateralCapCorporateGuarantee",
            "CollateralCapPersonalGuarantee",
            "HFSFlag",
            "SPVsCapitalization",
            "DeNuncementFlag",
            "LTV",
            "LTVRE",
            "ForbearanceMeasuresCount",
            "ForbearanceOfNPEFlag",
            "FPLUnderProbationFlag",
            "AgreementKey",
            "CREStocktakeFlag",
            "FINREPAssetClassSubClassCode",
            "FinancialDifficultyFlag",
            "InterestRate",
            "EffectiveInterestRate",
            "AcMaintenServiceBranch",
            "RelationshipUnit",
            "ConnectedBorrowerCategory",
            "ExposureMaturityDate",
            "ObligorLimitEndDate",
            "LitigationFlag",
            "FinRepCre",
            "BucketGt1Last12MFlag",
            "BucketGt1Last12MDate",
            "ObligorLimit",
            "SecuritizationCode",
            "MainBorrowerKey",
            "MainBorrowerTaxIdentificationNumber",
            "TepixCovidFlag",
            "NonCoreFlag",
            "PastDueDaysObligor",
            "PastDueDaysFacility",
            "DefaultStatus",
            "DefaultStatusReasonCode",
            "DefaultDate",
            "NPVLossOver1PercentFlag",
            "EliminatedEntityCode",
            "PBOriginationDate",
            "CPRSLevel1",
            "CPRSLevel2",
            "BusinessDate"
        ]

        UNION_FILE = load_landed_delta_table(
            spark,
            externalfiles_zone,  # type: ignore
            "extunionfile",
            union_file_final_filters,
            union_file_selected_fields,
        )

        # # ================== Portfolio ==================

        # portfolio_extra_filters = None

        # if portfolio_extra_filters is not None:
        #     portfolio_extra_filters.append(date_filter)
        #     portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        # else:
        #     portfolio_final_filters = date_filter

        # portfolio_selected_fields = [
        #     "PortfolioName",
        #     "Prefix",
        #     "PortfolioKey",
        # ]

        # portfolio = load_landed_portfolio_table(
        #     spark,
        #     curated_zone,  # type: ignore
        #     "portfolio",
        #     portfolio_selected_fields,
        # )

        # #-------SET THE CONDITIONS FOR JOINS
        # UnionFile_Portfolio_join_condition = (
        #     UNION_FILE["portfolio_code"]
        #     == portfolio["Prefix"]
        # )

        #  #-------APPLY THE JOINS
        # extunionfile_df = UNION_FILE.join(
        #     portfolio,
        #     UnionFile_Portfolio_join_condition,
        #     "left",
        # )

        # --------SET THE KEYS FOR HASHKEY BASED ON DATA MAPPING
        Unionfile_Id_primary_key_fields = [
            UNION_FILE.ExposureNumber,
            UNION_FILE.BorrowerKey,
            lit("ExternalFile"),
        ]

        Unionfile_Key_primary_key_fields = [
            UNION_FILE.ExposureNumber,
            UNION_FILE.BorrowerKey,
        ]

        # -------CREATION OF HASHKEY
        extunionfile_df = UNION_FILE.withColumn(
            "ExtUnionFileId",
            sha2(
                concat_ws(";", *Unionfile_Id_primary_key_fields),
                256,
            ),
        ).withColumn(
            "ExtUnionFileKey",
            concat_ws(";", *Unionfile_Key_primary_key_fields),
        ).withColumn(
            "SnapshotDate",
            when(
                length(col("SnapshotDate")) == 8,  # if format is yyyymmdd
                date_format(to_date(col("SnapshotDate"), "yyyyMMdd"), "yyyy-MM-dd")
            ).otherwise(
                col("SnapshotDate")  # assume it’s already in yyyy-mm-dd format
            )
        )

        # -------SELECTION OF FINAL COLUMNS
        result_df = extunionfile_df.select(
            extunionfile_df.ExtUnionFileId.alias("ExtUnionFileId"),
            lit("extfile").alias("SourceSystem"),
            lit("-9999").alias("PortfolioKey"),
            lit("Core Retail").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens"))
            .cast("timestamp")
            .alias("IngestionTimestamp"),
            lit("None").alias("BatchId"),
            extunionfile_df.ExtUnionFileKey.alias("ExtUnionFileKey"),
            UNION_FILE.Groupkey.alias("Groupkey"),
            UNION_FILE.GroupName.alias("GroupName"),
            UNION_FILE.BorrowerKey.alias("BorrowerKey"),
            UNION_FILE.BorrowerName.alias("BorrowerName"),
            UNION_FILE.BorrowerTaxIdentificationNumber.alias("BorrowerTaxIdentificationNumber"),
            UNION_FILE.MonitoringUnitCode.alias("MonitoringUnitCode"),
            UNION_FILE.ObligorRatingKey.alias("ObligorRatingKey"),
            UNION_FILE.ObligorRatingDate.alias("ObligorRatingDate"),
            UNION_FILE.ObligorCountry.alias("ObligorCountry"),
            UNION_FILE.UniqueExposureKey.alias("UniqueExposureKey"),
            UNION_FILE.ExposureKey.alias("ExposureKey"),
            UNION_FILE.ExposureNumber.alias("ExposureNumber"),
            UNION_FILE.ExposureSourceSystemKey.alias("ExposureSourceSystemKey"),
            UNION_FILE.ExposureCurrency.alias("ExposureCurrency"),
            UNION_FILE.ExposureSegmentation.alias("ExposureSegmentation"),
            UNION_FILE.ProductCode.alias("ProductCode"),
            UNION_FILE.ProductType.alias("ProductType"),
            UNION_FILE.IFRSGrossCarryingAmount.alias("IFRSGrossCarryingAmount"),
            UNION_FILE.SuspendedInterest.alias("SuspendedInterest"),
            UNION_FILE.AccruedSuspendedInterest.alias("AccruedSuspendedInterest"),
            UNION_FILE.AccruedInterest.alias("AccruedInterest"),
            UNION_FILE.OnbalanceAmount.alias("OnbalanceAmount"),
            UNION_FILE.TotalRemainingForAmortization.alias(
                "TotalRemainingForAmortization"
            ),
            UNION_FILE.BalanceOfPendingTransactions.alias(
                "BalanceOfPendingTransactions"
            ),
            UNION_FILE.OffBalanceAmount.alias("OffBalanceAmount"),
            UNION_FILE.UndrawnAmount.alias("UndrawnAmount"),
            UNION_FILE.ImpairedFlag.alias("ImpairedFlag"),
            UNION_FILE.ImpairmentType.alias("ImpairmentType"),
            UNION_FILE.IFRS9Stage.alias("IFRS9Stage"),
            UNION_FILE.ImpairmentAllowance.alias("ImpairmentAllowance"),
            UNION_FILE.FairValueAdjustmentsCR.alias("FairValueAdjustmentsCR"), 
            UNION_FILE.ProvisionsL3869.alias("ProvisionsL3869"),
            UNION_FILE.TotalProvisionsOnBalance.alias("TotalProvisionsOnBalance"),
            UNION_FILE.ProvisionsOffBalance.alias("ProvisionsOffBalance"),
            UNION_FILE.ProvisionsUndrawnLimits.alias("ProvisionsUndrawnLimits"),
            UNION_FILE.FairValueAdjustmentsFVPL.alias("FairValueAdjustmentsFVPL"),
            UNION_FILE.IFRS9ClassificationFlag.alias("IFRS9ClassificationFlag"),
            UNION_FILE.IFRS9POCIFlag.alias("IFRS9POCIFlag"),
            UNION_FILE.EliminationFlag.alias("EliminationFlag"),
            UNION_FILE.ServiceType.alias("ServiceTypeCode"),
            UNION_FILE.ServiceType.alias("ServiceType"),
            UNION_FILE.BookingEntity.alias("BookingEntity"),
            UNION_FILE.OriginatingEntity.alias("OriginatingEntity"),
            UNION_FILE.DaysPastDue.alias("DaysPastDue"),
            UNION_FILE.CurrentAmount.alias("CurrentAmount"),
            UNION_FILE.DelinquentAmount.alias("DelinquentAmount"),
            UNION_FILE.DelinquentAmountBucket1.alias("DelinquentAmountBucket1"),
            UNION_FILE.DelinquentAmountBucket2.alias("DelinquentAmountBucket2"),
            UNION_FILE.DelinquentAmountBucket3.alias("DelinquentAmountBucket3"),
            UNION_FILE.DelinquentAmountBucket4.alias("DelinquentAmountBucket4"),
            UNION_FILE.DelinquentAmountBucket5.alias("DelinquentAmountBucket5"),
            UNION_FILE.DelinquentAmountBucket6.alias("DelinquentAmountBucket6"),
            UNION_FILE.DelinquentAmountBucket7.alias("DelinquentAmountBucket7"),
            UNION_FILE.DelinquentAmountBucket8.alias("DelinquentAmountBucket8"),
            UNION_FILE.DelinquentAmountBucket9.alias("DelinquentAmountBucket9"),
            UNION_FILE.DelinquentAmountBucket10.alias("DelinquentAmountBucket10"),
            UNION_FILE.DelinquentAmountBucket11.alias("DelinquentAmountBucket11"),
            UNION_FILE.DelinquentAmountBucket12.alias("DelinquentAmountBucket12"),
            UNION_FILE.DelinquentAmountBucket13.alias("DelinquentAmountBucket13"),
            UNION_FILE.DelinquentAmountBucket14.alias("DelinquentAmountBucket14"),
            UNION_FILE.BucketCategoryNMateriality.alias(
                "BucketCategoryNMateriality"
            ),
            UNION_FILE.BucketCategoryWithMateriality.alias(
                "BucketCategoryWithMateriality"
            ),
            UNION_FILE.NPLNoMaterialityFlag.alias("NPLNoMaterialityFlag"),
            UNION_FILE.NPLWithMaterialityFlag.alias("NPLWithMaterialityFlag"),
            UNION_FILE.EBAStatus.alias("EBAStatus"),
            UNION_FILE.UTPIndicator.alias("UTPTypeCode"),
            UNION_FILE.UTPIndicator.alias("UTPType"),
            UNION_FILE.ProbationStartDate.alias("ProbationStartDate"),
            UNION_FILE.ECAAssetClass.alias("ECAAssetClass"),
            UNION_FILE.FINREPAssetClass.alias("FINREPAssetClassCode"),
            UNION_FILE.FINREPAssetClass.alias("FINREPAssetClass"),
            UNION_FILE.IFRSAssetClass.alias("IFRSAssetClass"),
            UNION_FILE.ForbearanceType.alias("ForbearanceTypeCode"),
            UNION_FILE.ForbearanceType.alias("ForbearanceType"),
            UNION_FILE.ForbearanceDate.alias("ForbearanceDate"),
            UNION_FILE.NaceCode.alias("NaceCode"),
            UNION_FILE.ECAIndustry.alias("ECAIndustry"),
            UNION_FILE.FINREPIndustry.alias("FINREPIndustry"),
            UNION_FILE.IFRSIndustry.alias("IFRSIndustry"),
            UNION_FILE.ECAPublicFlag.alias("ECAPublicFlag"),
            UNION_FILE.StateGuaranteeFlag.alias("StateGuaranteeFlag"),
            UNION_FILE.CollateralCapCash.alias("CollateralCapCash"),
            UNION_FILE.CollateralCapRRE.alias("CollateralCapRRE"),
            UNION_FILE.CollateralCapCRE.alias("CollateralCapCRE"),
            UNION_FILE.CollateralCapORE.alias("CollateralCapORE"),
            UNION_FILE.CollateralCapVessels.alias("CollateralCapVessels"),
            UNION_FILE.CollateralCapTbills.alias("CollateralCapTbills"),
            UNION_FILE.CollateralCapGGG.alias("CollateralCapGGG"),
            UNION_FILE.CollateralCapStateRec.alias("CollateralCapStateRec"),
            UNION_FILE.CollateralCapInventory.alias("CollateralCapInventory"),
            UNION_FILE.CollateralCapAssRec.alias("CollateralCapAssRec"),
            UNION_FILE.CollateralCapIndustrial.alias("CollateralCapIndustrial"),
            UNION_FILE.CollateralCapOtherCollateral.alias("CollateralCapOtherCollateral"),
            UNION_FILE.CollateralCapOtherFinGuarantee.alias("CollateralCapOtherFinGuarantee"),
            UNION_FILE.CollateralCapTotalTangible.alias("CollateralCapTotalTangible"),
            UNION_FILE.CollateralCapCorporateGuarantee.alias("CollateralCapCorporateGuarantee"),
            UNION_FILE.CollateralCapPersonalGuarantee.alias("CollateralCapPersonalGuarantee"),
            UNION_FILE.HFSFlag.alias("HFSFlag"),
            UNION_FILE.SPVsCapitalization.alias("SPVsCapitalization"),
            UNION_FILE.DeNuncementFlag.alias("DeNuncementFlag"),
            UNION_FILE.LTV.alias("LTV"),
            UNION_FILE.LTVRE.alias("LTVRE"),
            UNION_FILE.ForbearanceMeasuresCount.alias("ForbearanceMeasuresCount"),
            UNION_FILE.ForbearanceOfNPEFlag.alias("ForbearanceOfNPEFlag"),
            UNION_FILE.FPLUnderProbationFlag.alias("FPLUnderProbationFlag"),
            UNION_FILE.AgreementKey.alias("AgreementKey"),
            UNION_FILE.CREStocktakeFlag.alias("CREStocktakeFlag"),
            UNION_FILE.FINREPAssetClassSubClassCode.alias(
                "FINREPAssetClassSubClassCode"
            ),
            UNION_FILE.FinancialDifficultyFlag.alias("FinancialDifficultyFlag"),
            UNION_FILE.InterestRate.alias("InterestRate"),
            UNION_FILE.EffectiveInterestRate.alias("EffectiveInterestRate"),
            UNION_FILE.AcMaintenServiceBranch.alias(
                "AcMaintenServiceBranchCode"
            ),
            UNION_FILE.AcMaintenServiceBranch.alias("AcMaintenServiceBranch"),
            UNION_FILE.RelationshipUnit.alias("RelationshipUnitCode"),
            UNION_FILE.RelationshipUnit.alias("RelationshipUnit"),
            UNION_FILE.ConnectedBorrowerCategory.alias(
                "ConnectedBorrowerCategoryCode"
            ),
            UNION_FILE.ConnectedBorrowerCategory.alias("ConnectedBorrowerCategory"),
            UNION_FILE.ExposureMaturityDate.alias("ExposureMaturityDate"),
            UNION_FILE.ObligorLimitEndDate.alias("ObligorLimitEndDate"),
            UNION_FILE.LitigationFlag.alias("LitigationFlag"),
            UNION_FILE.FinRepCre.alias("FinRepCre"),
            UNION_FILE.BucketGt1Last12MFlag.alias("BucketGt1Last12MFlag"),
            UNION_FILE.BucketGt1Last12MDate.alias("BucketGt1Last12MDate"),
            UNION_FILE.ObligorLimit.alias("ObligorLimit"),
            UNION_FILE.SecuritizationCode.alias("SecuritizationCode"),
            UNION_FILE.MainBorrowerKey.alias("MainBorrowerKey"),
            UNION_FILE.MainBorrowerTaxIdentificationNumber.alias(
                "MainBorrowerTaxIdentificationNumber"
            ),
            UNION_FILE.TepixCovidFlag.alias("TepixCovidFlag"),
            UNION_FILE.NonCoreFlag.alias("NonCoreFlag"),
            UNION_FILE.PastDueDaysObligor.alias("PastDueDaysObligor"),
            UNION_FILE.PastDueDaysFacility.alias("PastDueDaysFacility"),
            UNION_FILE.DefaultStatus.alias("DefaultStatus"),
            UNION_FILE.DefaultStatusReasonCode.alias("DefaultStatusReasonCode"),
            UNION_FILE.DefaultStatusReasonCode.alias("DefaultStatusReason"),
            UNION_FILE.DefaultDate.alias("DefaultDate"),
            UNION_FILE.NPVLossOver1PercentFlag.alias(
                "NPVLossOver1PercentFlag"
            ),
            UNION_FILE.EliminatedEntityCode.alias("EliminatedEntityCode"),
            UNION_FILE.PBOriginationDate.alias("PBOriginationDate"),
            UNION_FILE.CPRSLevel1.alias("CPRSLevel1"),
            UNION_FILE.CPRSLevel2.alias("CPRSLevel2"),
            extunionfile_df.SnapshotDate.cast("date").alias("SnapshotDate"),
            UNION_FILE.BusinessDate.alias("BusinessDate"),
        )
        # --------------APPEND DATA TO CURATED
        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                externalfiles_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_ext_union_file_extfile(spark, curated_zone, catalog_landed, reference_zone, externalfiles_zone, run_date, schema, env_prefix)